In [ ]:
# import numpy as np

# # Function
# def f(x):
#     return np.exp(-x**2)

# # Trapezoid rule
# def trapezoid(f, a, b, n):
#     x = np.linspace(a, b, n+1)
#     y = f(x)
#     dx = (b - a)/n
#     return (dx/2) * np.sum(y[:-1] + y[1:])

# # Adaptive trapezoid
# def adaptive_trapezoid(f, a, b, tol=1e-10):
#     n = 2
#     prev = trapezoid(f, a, b, n)
#     steps = [(n, prev)]
#     while True:
#         n *= 2
#         curr = trapezoid(f, a, b, n)
#         steps.append((n, curr))
#         if abs(curr - prev) < tol:
#             return curr, steps
#         prev = curr

# # Parameters
# a, b = 0, 1
# I_exact = 0.746824132812  # reference value

# # Fixed trapezoid results
# n_values = [2, 4, 8, 16, 32, 64, 128, 256]
# print("=== Trapezoid Rule (Fixed n) ===")
# for n in n_values:
#     I_n = trapezoid(f, a, b, n)
#     error = abs(I_n - I_exact)
#     print(f"n={n:4d} | Result={I_n:.12f} | Error={error:.2e}")

# # Adaptive trapezoid results
# print("\n=== Adaptive Trapezoid Refinement ===")
# I_adapt, steps = adaptive_trapezoid(f, a, b, tol=1e-10)
# for n, val in steps:
#     error = abs(val - I_exact)
#     print(f"n={n:4d} | Result={val:.12f} | Error={error:.2e}")

# print(f"\nFinal Adaptive Result = {I_adapt:.12f}")
# print(f"Exact Analytical Value = {I_exact:.12f}")


In [ ]:
import ROOT
import numpy as np

def RosenBrock(vecx):
    x = vecx[0]
    y = vecx[1]
    return (1 - x)**2 + 100 * (y - x**2)**2  # Correct Rosenbrock function

# create minimizer giving a name and a name (optionally) for the specific algorithm
#  possible choices are:
#     minimizerName                  algoName
#
#     Minuit                     Migrad, Simplex,Combined,Scan  (default is Migrad)
#     Minuit2                    Migrad, BFGS, Simplex,Combined,Scan  (default is Migrad)
#     GSLMultiMin                ConjugateFR, ConjugatePR, BFGS, BFGS2, SteepestDescent
#     GSLSimAn
#     Genetic

def NumericalMinimization(minimizerName="Minuit2",
                          algoName="Migrad",
                          randomSeed=-1):
    
    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if (not minimizer):
        raise RuntimeError(
            "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))

    # Set tolerance and other minimizer parameters, one can also use default
    # values

    minimizer.SetMaxFunctionCalls(1000000)  # working for Minuit/Minuit2
    # for GSL minimizers - no effect in Minuit/Minuit2
    minimizer.SetMaxIterations(10000)
    minimizer.SetTolerance(0.001)
    minimizer.SetPrintLevel(1)

    # Create function wrapper for minimizer

    f = ROOT.Math.Functor(RosenBrock, 2)

    # Starting point
    variable = [-1., 1.2]
    step = [0.01, 0.01]
    if (randomSeed >= 0):
        r = ROOT.TRandom2(randomSeed)
        variable[0] = r.Uniform(-20, 20)
        variable[1] = r.Uniform(-20, 20)

    minimizer.SetFunction(f)

    # Set the free variables to be minimized !
    minimizer.SetVariable(0, "x", variable[0], step[0])
    minimizer.SetVariable(1, "y", variable[1], step[1])

    # Do the minimization
    ret = minimizer.Minimize()

    xs = minimizer.X()
    print("Minimum: f({} , {}) = {}".format(xs[0], xs[1], minimizer.MinValue()))

    # Expected minimum is f(1,1) = 0
    expected_min = 0.0
    tolerance = 1.E-4
    if (ret and abs(minimizer.MinValue() - expected_min) < tolerance):
        print("Minimizer {} - {} converged to the right minimum!".format(minimizerName, algoName))
    else:
        print("Minimizer {} - {} failed to converge! Found minimum at f({}, {}) = {}".format(
            minimizerName, algoName, xs[0], xs[1], minimizer.MinValue()))
        # Don't raise an error for demonstration, just print warning

if __name__ == "__main__":
    NumericalMinimization()

In [ ]:
# import numpy as np
# from scipy.optimize import curve_fit
# from scipy.stats import t

# # Example model
# def model(x, a, b, c, d):
#     return a * np.exp(-b * x) + c * x + d

# # Fake data
# xdata = np.linspace(0, 4, 50)
# y = model(xdata, 2.5, 1.3, 0.5, 1.0)
# rng = np.random.default_rng(0)
# y_noise = y + 0.2 * rng.normal(size=len(xdata))

# # Fit
# popt, pcov = curve_fit(model, xdata, y_noise)

# # Extract standard errors
# perr = np.sqrt(np.diag(pcov))

# # 90% confidence intervals
# alpha = 0.10
# dof = max(0, len(xdata) - len(popt))  # degrees of freedom
# tval = t.ppf(1.0 - alpha/2., dof)

# ci = [ (p - tval*e, p + tval*e) for p, e in zip(popt, perr) ]

# print("Parameters:", popt)
# print("90% CI:", ci)


In [ ]:
# import numpy as np
# from iminuit import Minuit
# from iminuit.cost import LeastSquares

# # Example model
# def model(x, a, b, c, d):
#     return a * np.exp(-b * x) + c * x + d

# # Fake data
# x = np.linspace(0, 4, 50)
# y = model(x, 2.5, 1.3, 0.5, 1.0) + 0.2 * np.random.normal(size=len(x))
# yerr = np.full_like(y, 0.2)

# # Least squares cost
# cost = LeastSquares(x, y, yerr, model)
# m = Minuit(cost, a=1, b=1, c=1, d=1)  # initial guesses
# m.migrad()  # minimize
# m.hesse()   # covariance matrix

# print(m.values)   # best-fit params
# print(m.errors)   # 1σ errors (~68%)

# # For 90% CI, use m.mnprofile or m.mncontour
# for name in m.parameters:
#     ci90 = m.draw_mnprofile(name)  # 90% ≈ 1.64σ
#     # print(name, "90% CI:", ci90)


In [ ]:
# import ROOT
# import numpy as np

# # Sample data for chi-squared calculation
# # You can replace this with your actual data
# x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
# y_data = np.array([2.1, 3.9, 6.1, 8.0, 9.9])
# y_errors = np.array([0.2, 0.3, 0.2, 0.4, 0.3])

# def linear_model(x, params):
#     """Linear model: y = a*x + b"""
#     a, b = params[0], params[1]
#     return a * x + b

# def chi2_function(params):
#     """
#     Calculate chi-squared for a linear fit
#     params[0] = slope (a)
#     params[1] = intercept (b)
#     """
#     chi2 = 0.0
#     for i in range(len(x_data)):
#         predicted = linear_model(x_data[i], params)
#         residual = y_data[i] - predicted
#         chi2 += (residual / y_errors[i])**2
#     return chi2

# # Alternative: Generic chi2 function for any model
# def generic_chi2_function(params):
#     """
#     Generic chi-squared function - modify this for your specific model
#     """
#     # Example: quadratic model y = a*x^2 + b*x + c
#     # Uncomment and modify as needed:
    
#     # chi2 = 0.0
#     # for i in range(len(x_data)):
#     #     predicted = params[0]*x_data[i]**2 + params[1]*x_data[i] + params[2]
#     #     residual = y_data[i] - predicted
#     #     chi2 += (residual / y_errors[i])**2
#     # return chi2
    
#     # For now, use the linear model
#     return chi2_function(params)

# def NumericalMinimization(minimizerName="Minuit2",
#                           algoName="",
#                           randomSeed=-1):
    
#     minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
#     if (not minimizer):
#         raise RuntimeError(
#             "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))
    
#     # Set tolerance and other minimizer parameters
#     minimizer.SetMaxFunctionCalls(1000000)
#     minimizer.SetMaxIterations(10000)
#     minimizer.SetTolerance(0.001)
#     minimizer.SetPrintLevel(1)
    
#     # Create function wrapper for minimizer
#     # Use 2 parameters for linear fit (slope and intercept)
#     f = ROOT.Math.Functor(chi2_function, 2)
    
#     # Starting point for parameters [slope, intercept]
#     # You can estimate these from your data or use reasonable guesses
#     variable = [1.0, 0.0]  # Initial guess: slope=1, intercept=0
#     step = [0.01, 0.01]    # Step sizes for parameters
    
#     if (randomSeed >= 0):
#         r = ROOT.TRandom2(randomSeed)
#         variable[0] = r.Uniform(-10, 10)  # Random slope
#         variable[1] = r.Uniform(-10, 10)  # Random intercept
    
#     minimizer.SetFunction(f)
    
#     # Set the free variables to be minimized
#     minimizer.SetVariable(0, "slope", variable[0], step[0])
#     minimizer.SetVariable(1, "intercept", variable[1], step[1])
    
#     # Optional: Set parameter limits if needed
#     # minimizer.SetVariableLimits(0, -100, 100)  # Limit slope
#     # minimizer.SetVariableLimits(1, -100, 100)  # Limit intercept
    
#     # Do the minimization
#     ret = minimizer.Minimize()
    
#     xs = minimizer.X()
#     chi2_min = minimizer.MinValue()
    
#     print("Minimum: chi2(slope={:.4f}, intercept={:.4f}) = {:.4f}".format(
#         xs[0], xs[1], chi2_min))
    
#     # Calculate degrees of freedom
#     ndf = len(x_data) - 2  # n_data_points - n_parameters
#     reduced_chi2 = chi2_min / ndf
    
#     print("Reduced chi2 = {:.4f}".format(reduced_chi2))
#     print("Number of degrees of freedom = {}".format(ndf))
    
#     # Check convergence
#     if ret:
#         print("Minimizer {} - {} converged successfully!".format(minimizerName, algoName))
        
#         # Print parameter errors if available
#         if minimizer.Errors():
#             errors = minimizer.Errors()
#             print("Parameter errors:")
#             print("  slope error = {:.4f}".format(errors[0]))
#             print("  intercept error = {:.4f}".format(errors[1]))
#     else:
#         print("Minimizer {} - {} failed to converge !!!".format(minimizerName, algoName))
#         raise RuntimeError("NumericalMinimization failed to converge!")
    
#     return xs, chi2_min, reduced_chi2

# def plot_results(params):
#     """Optional: Plot the data and fitted curve"""
#     try:
#         import matplotlib.pyplot as plt
        
#         # Generate points for the fitted line
#         x_fit = np.linspace(min(x_data), max(x_data), 100)
#         y_fit = linear_model(x_fit, params)
        
#         plt.figure(figsize=(8, 6))
#         plt.errorbar(x_data, y_data, yerr=y_errors, fmt='o', label='Data')
#         plt.plot(x_fit, y_fit, 'r-', label=f'Fit: y = {params[0]:.3f}x + {params[1]:.3f}')
#         plt.xlabel('x')
#         plt.ylabel('y')
#         plt.legend()
#         plt.grid(True)
#         plt.title('Chi-squared Fit Results')
#         plt.show()
        
#     except ImportError:
#         print("matplotlib not available for plotting")

# if __name__ == "__main__":
#     # Run the minimization
#     fitted_params, min_chi2, reduced_chi2 = NumericalMinimization()
    
#     # Optionally plot results
#     # plot_results(fitted_params)

In [ ]:
import ROOT
import math
import numpy as np

def generic_function(x, par):
    """
    Generic function to fit
    f(x) = par[0] * sin(par[1]*x) + par[2] * exp(par[3]*x) + par[4]
    """
    return par[0] * math.sin(par[1] * x[0]) + par[2] * math.exp(par[3] * x[0]) + par[4]

def create_sample_data():
    """Create sample data with some noise"""
    x = np.linspace(0, 10, 50)
    # True function: f(x) = 2.5 * sin(0.8*x) + 1.2 * exp(-0.3*x) + 0.5
    y_true = 2.5 * np.sin(0.8*x) + 1.2 * np.exp(-0.3*x) + 0.5
    # Add some Gaussian noise
    y_data = y_true + 0.3 * np.random.normal(size=len(x))
    return x, y_data, y_true

def main():
    # Create sample data
    x_data, y_data, y_true = create_sample_data()
    
    # Create a ROOT graph with the data
    graph = ROOT.TGraphErrors(len(x_data))
    
    for i, (x, y) in enumerate(zip(x_data, y_data)):
        graph.SetPoint(i, x, y)
        graph.SetPointError(i, 0, 0.3)  # Assuming constant error of 0.3
    
    # Create the fit function using TF1
    fit_function = ROOT.TF1("fit_function", generic_function, 0, 10, 5)
    
    # Set initial parameter values and names
    fit_function.SetParName(0, "Amplitude")
    fit_function.SetParName(1, "Frequency")
    fit_function.SetParName(2, "ExpAmplitude")
    fit_function.SetParName(3, "ExpDecay")
    fit_function.SetParName(4, "Offset")
    
    # Set initial parameter guesses
    fit_function.SetParameter(0, 2.0)   # Amplitude guess
    fit_function.SetParameter(1, 1.0)   # Frequency guess
    fit_function.SetParameter(2, 1.0)   # Exponential amplitude guess
    fit_function.SetParameter(3, -0.5)  # Exponential decay guess
    fit_function.SetParameter(4, 0.0)   # Offset guess
    
    # Set parameter limits (optional but often helpful)
    fit_function.SetParLimits(0, 0.1, 5.0)    # Amplitude between 0.1 and 5
    fit_function.SetParLimits(1, 0.1, 2.0)    # Frequency between 0.1 and 2
    fit_function.SetParLimits(3, -2.0, 0.0)   # Decay rate negative
    
    # Perform the least squares fit
    print("Performing least squares fit...")
    fit_result = graph.Fit(fit_function, "S")  # "S" saves the fit result
    
    # Print fit results
    print("\n=== Fit Results ===")
    print(f"Fit status: {fit_result.Status()}")
    print(f"Chi-squared: {fit_function.GetChisquare():.4f}")
    print(f"NDF: {fit_function.GetNDF()}")
    print(f"Chi-squared/NDF: {fit_function.GetChisquare()/fit_function.GetNDF():.4f}")
    
    print("\n=== Parameter Values ===")
    for i in range(5):
        par_name = fit_function.GetParName(i)
        par_value = fit_function.GetParameter(i)
        par_error = fit_function.GetParError(i)
        print(f"{par_name}: {par_value:.4f} ± {par_error:.4f}")
    
    # Create a canvas and draw the results
    canvas = ROOT.TCanvas("canvas", "Least Squares Fit Example", 800, 600)
    
    # Draw the graph and fit
    graph.SetTitle("Least Squares Fit with Generic Function")
    graph.GetXaxis().SetTitle("x")
    graph.GetYaxis().SetTitle("y")
    graph.SetMarkerStyle(20)
    graph.SetMarkerSize(0.8)
    graph.Draw("AP")
    
    # Draw the fit function
    fit_function.SetLineColor(ROOT.kRed)
    fit_function.SetLineWidth(2)
    fit_function.Draw("same")
    
    # Add a legend
    legend = ROOT.TLegend(0.7, 0.7, 0.9, 0.9)
    legend.AddEntry(graph, "Data", "p")
    legend.AddEntry(fit_function, "Fit", "l")
    legend.Draw()
    
    canvas.Update()
    canvas.Draw()

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Exemplo de uso do método LeastSquareFit() da classe ROOT::Fit::Fitter
usando PyROOT para fazer um ajuste de mínimos quadrados em dados binados.
"""

import ROOT
import numpy as np

def main():
    # Configuração do estilo
    ROOT.gStyle.SetOptFit(1111)
    ROOT.gStyle.SetOptStat(1111)
    
    # 1. Criar dados simulados (distribuição gaussiana + ruído)
    print("=" * 60)
    print("Exemplo de Least Square Fit com ROOT::Fit::Fitter")
    print("=" * 60)
    
    # Parâmetros da gaussiana verdadeira
    true_mean = 5.0
    true_sigma = 1.5
    true_amplitude = 1000.0
    
    # Criar histograma para armazenar os dados
    nbins = 50
    xmin, xmax = 0.0, 10.0
    h1 = ROOT.TH1D("h1", "Dados para Least Square Fit;x;Contagens", 
                   nbins, xmin, xmax)
    
    # Preencher o histograma com dados gaussianos
    np.random.seed(42)
    data = np.random.normal(true_mean, true_sigma, int(true_amplitude))
    for value in data:
        h1.Fill(value)
    
    print(f"\nDados gerados:")
    print(f"  - Média verdadeira: {true_mean}")
    print(f"  - Sigma verdadeiro: {true_sigma}")
    print(f"  - Número de eventos: {h1.GetEntries()}")
    
    # 2. Preparar o BinData para o fit
    # Converter o histograma em um objeto BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, h1)
    
    print(f"\nBinData criado com {bin_data.Size()} pontos")
    
    # 3. Definir a função modelo (gaussiana)
    # f(x) = p[0] * exp(-0.5 * ((x - p[1]) / p[2])^2)
    func = ROOT.TF1("gaus", "gaus", xmin, xmax)
    
    # Valores iniciais dos parâmetros
    func.SetParameter(0, 100.0)   # Amplitude inicial
    func.SetParameter(1, 5.0)     # Média inicial
    func.SetParameter(2, 1.0)     # Sigma inicial
    
    # Criar um wrapper IParamFunction
    wrapped_func = ROOT.Math.WrappedTF1(func)
    
    print("\nParâmetros iniciais:")
    for i in range(func.GetNpar()):
        print(f"  p[{i}] = {func.GetParameter(i):.4f}")
    
    # 4. Configurar e executar o Fitter
    fitter = ROOT.Fit.Fitter()
    
    # Configurar o fit
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(1)
    
    # Definir a função modelo
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    print("\n" + "=" * 60)
    print("Executando LeastSquareFit...")
    print("=" * 60)
    
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # 5. Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS DO FIT")
    print("=" * 60)
    
    if fit_result:
        result = fitter.Result()
        print(f"\nStatus do fit: {'Sucesso' if result.IsValid() else 'Falhou'}")
        print(f"Chi2/NDF: {result.Chi2():.4f} / {result.Ndf()}")
        print(f"Chi2/NDF reduzido: {result.Chi2()/result.Ndf():.4f}")
        print(f"Probabilidade: {result.Prob():.6f}")
        
        print("\nParâmetros ajustados:")
        for i in range(result.NPar()):
            print(f"  p[{i}] = {result.Parameter(i):.4f} ± {result.ParError(i):.4f}")
        
        # Atualizar a função TF1 com os parâmetros ajustados
        for i in range(result.NPar()):
            func.SetParameter(i, result.Parameter(i))
            func.SetParError(i, result.ParError(i))
        
        # 6. Visualizar o resultado
        # canvas = ROOT.TCanvas("canvas", "Least Square Fit Result", 800, 600)
        # canvas.SetGrid()
        
        # h1.SetLineColor(ROOT.kBlue)
        # h1.SetLineWidth(2)
        # h1.Draw("E")
        
        # func.SetLineColor(ROOT.kRed)
        # func.SetLineWidth(2)
        # func.Draw("SAME")
        
        # # Adicionar legenda
        # legend = ROOT.TLegend(0.15, 0.65, 0.45, 0.88)
        # legend.SetBorderSize(1)
        # legend.SetFillColor(0)
        # legend.AddEntry(h1, "Dados", "lep")
        # legend.AddEntry(func, "Fit: Gaussiana", "l")
        # legend.AddEntry(ROOT.nullptr, 
        #                f"#chi^{{2}}/NDF = {result.Chi2()/result.Ndf():.3f}", "")
        # legend.AddEntry(ROOT.nullptr, 
        #                f"#mu = {result.Parameter(1):.3f} #pm {result.ParError(1):.3f}", "")
        # legend.AddEntry(ROOT.nullptr, 
        #                f"#sigma = {result.Parameter(2):.3f} #pm {result.ParError(2):.3f}", "")
        # legend.Draw()
        
        # canvas.Update()
        
        # # Salvar o resultado
        # canvas.SaveAs("leastsquare_fit_result.png")
        # print("\nGráfico salvo como: leastsquare_fit_result.png")
        
        # # Manter a janela aberta
        # print("\nPressione Enter para sair...")
        # input()
    else:
        print("\nErro: O fit falhou!")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Exemplo de uso do método LeastSquareFit() da classe ROOT::Fit::Fitter
usando PyROOT para fazer um ajuste de mínimos quadrados em dados binados.
"""

import ROOT
import numpy as np

def gaussian_model(x, par):
    """
    Função modelo: Gaussiana
    f(x) = par[0] * exp(-0.5 * ((x - par[1]) / par[2])^2)
    
    Args:
        x: array com coordenadas x
        par: array com parâmetros [amplitude, mean, sigma]
    
    Returns:
        Valor da função gaussiana
    """
    arg = 0.0
    if par[2] != 0:
        arg = (x[0] - par[1]) / par[2]
    
    return par[0] * ROOT.TMath.Exp(-0.5 * arg * arg)

def main():
    # Configuração do estilo
    ROOT.gStyle.SetOptFit(1111)
    ROOT.gStyle.SetOptStat(1111)
    
    # 1. Criar dados simulados (distribuição gaussiana + ruído)
    print("=" * 60)
    print("Exemplo de Least Square Fit com ROOT::Fit::Fitter")
    print("=" * 60)
    
    # Parâmetros da gaussiana verdadeira
    true_mean = 5.0
    true_sigma = 1.5
    true_amplitude = 1000.0
    
    # Criar histograma para armazenar os dados
    nbins = 50
    xmin, xmax = 0.0, 10.0
    h1 = ROOT.TH1D("h1", "Dados para Least Square Fit;x;Contagens", 
                   nbins, xmin, xmax)
    
    # Preencher o histograma com dados gaussianos
    np.random.seed(42)
    data = np.random.normal(true_mean, true_sigma, int(true_amplitude))
    for value in data:
        h1.Fill(value)
    
    print(f"\nDados gerados:")
    print(f"  - Média verdadeira: {true_mean}")
    print(f"  - Sigma verdadeiro: {true_sigma}")
    print(f"  - Número de eventos: {h1.GetEntries()}")
    
    # 2. Preparar o BinData para o fit
    # Converter o histograma em um objeto BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, h1)
    
    print(f"\nBinData criado com {bin_data.Size()} pontos")
    
    # 3. Definir a função modelo (gaussiana)
    # f(x) = p[0] * exp(-0.5 * ((x - p[1]) / p[2])^2)
    func = ROOT.TF1("gaus", gaussian_model, xmin, xmax, 3)
    
    # Valores iniciais dos parâmetros
    func.SetParameter(0, 100.0)   # Amplitude inicial
    func.SetParameter(1, 5.0)     # Média inicial
    func.SetParameter(2, 1.0)     # Sigma inicial
    
    # Criar um wrapper IParamFunction
    wrapped_func = ROOT.Math.WrappedTF1(func)
    
    print("\nParâmetros iniciais:")
    for i in range(func.GetNpar()):
        print(f"  p[{i}] = {func.GetParameter(i):.4f}")
    
    # 4. Configurar e executar o Fitter
    fitter = ROOT.Fit.Fitter()
    
    # Configurar o fit
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(1)
    
    # Definir a função modelo
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    print("\n" + "=" * 60)
    print("Executando LeastSquareFit...")
    print("=" * 60)
    
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # 5. Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS DO FIT")
    print("=" * 60)
    
    if fit_result:
        result = fitter.Result()
        print(f"\nStatus do fit: {'Sucesso' if result.IsValid() else 'Falhou'}")
        print(f"Chi2/NDF: {result.Chi2():.4f} / {result.Ndf()}")
        print(f"Probabilidade: {result.Prob():.6f}")
        
        print("\nParâmetros ajustados:")
        for i in range(result.NPar()):
            print(f"  p[{i}] = {result.Parameter(i):.4f} ± {result.ParError(i):.4f}")
        
        # Atualizar a função TF1 com os parâmetros ajustados
        for i in range(result.NPar()):
            func.SetParameter(i, result.Parameter(i))
            func.SetParError(i, result.ParError(i))
        
        # 6. Visualizar o resultado
        canvas = ROOT.TCanvas("canvas", "Least Square Fit Result", 800, 600)
        canvas.SetGrid()
        
        h1.SetLineColor(ROOT.kBlue)
        h1.SetLineWidth(2)
        h1.Draw("E")
        
        func.SetLineColor(ROOT.kRed)
        func.SetLineWidth(2)
        func.Draw("SAME")
        
        # Adicionar legenda
        legend = ROOT.TLegend(0.15, 0.65, 0.45, 0.88)
        legend.SetBorderSize(1)
        legend.SetFillColor(0)
        legend.AddEntry(h1, "Dados", "lep")
        legend.AddEntry(func, "Fit: Gaussiana", "l")
        legend.AddEntry(ROOT.nullptr, 
                       f"#chi^{{2}}/NDF = {result.Chi2()/result.Ndf():.3f}", "")
        legend.AddEntry(ROOT.nullptr, 
                       f"#mu = {result.Parameter(1):.3f} #pm {result.ParError(1):.3f}", "")
        legend.AddEntry(ROOT.nullptr, 
                       f"#sigma = {result.Parameter(2):.3f} #pm {result.ParError(2):.3f}", "")
        legend.Draw()
        
        canvas.Update()
        
        # Salvar o resultado
        canvas.SaveAs("leastsquare_fit_result.png")
        print("\nGráfico salvo como: leastsquare_fit_result.png")
        
        # Manter a janela aberta
        print("\nPressione Enter para sair...")
        input()
    else:
        print("\nErro: O fit falhou!")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Exemplo de uso do método LeastSquareFit() da classe ROOT::Fit::Fitter
usando PyROOT para fazer um ajuste de mínimos quadrados em dados binados.
"""

import ROOT
import numpy as np

def gaussian_model(x, par):
    """
    Função modelo: Gaussiana
    f(x) = par[0] * exp(-0.5 * ((x - par[1]) / par[2])^2)
    
    Args:
        x: array com coordenadas x
        par: array com parâmetros [amplitude, mean, sigma]
    
    Returns:
        Valor da função gaussiana
    """
    arg = 0.0
    if par[2] != 0:
        arg = (x[0] - par[1]) / par[2]
    
    return par[0] * ROOT.TMath.Exp(-0.5 * arg * arg)

def main():
    # Configuração do estilo
    ROOT.gStyle.SetOptFit(1111)
    ROOT.gStyle.SetOptStat(1111)
    
    # 1. Criar dados simulados (distribuição gaussiana + ruído)
    print("=" * 60)
    print("Exemplo de Least Square Fit com ROOT::Fit::Fitter")
    print("=" * 60)
    
    # Parâmetros da gaussiana verdadeira
    true_mean = 5.0
    true_sigma = 1.5
    true_amplitude = 1000.0
    
    # Criar histograma para armazenar os dados
    nbins = 50
    xmin, xmax = 0.0, 10.0
    h1 = ROOT.TH1D("h1", "Dados para Least Square Fit;x;Contagens", 
                   nbins, xmin, xmax)
    
    # Preencher o histograma com dados gaussianos
    np.random.seed(42)
    data = np.random.normal(true_mean, true_sigma, int(true_amplitude))
    for value in data:
        h1.Fill(value)
    
    print(f"\nDados gerados:")
    print(f"  - Média verdadeira: {true_mean}")
    print(f"  - Sigma verdadeiro: {true_sigma}")
    print(f"  - Número de eventos: {h1.GetEntries()}")
    
    # 2. Preparar o BinData para o fit
    # Converter o histograma em um objeto BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, h1)
    
    print(f"\nBinData criado com {bin_data.Size()} pontos")
    
    # 3. Definir a função modelo (gaussiana)
    # f(x) = p[0] * exp(-0.5 * ((x - p[1]) / p[2])^2)
    func = ROOT.TF1("gaus", gaussian_model, xmin, xmax, 3)
    
    # Valores iniciais dos parâmetros
    func.SetParameter(0, 100.0)   # Amplitude inicial
    func.SetParameter(1, 5.0)     # Média inicial
    func.SetParameter(2, 1.0)     # Sigma inicial
    
    # Criar um wrapper IParamFunction
    wrapped_func = ROOT.Math.WrappedTF1(func)
    
    print("\nParâmetros iniciais:")
    for i in range(func.GetNpar()):
        print(f"  p[{i}] = {func.GetParameter(i):.4f}")
    
    # 4. Configurar e executar o Fitter
    fitter = ROOT.Fit.Fitter()
    
    # Configurar o fit
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(1)
    
    # Definir a função modelo
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    print("\n" + "=" * 60)
    print("Executando LeastSquareFit...")
    print("=" * 60)
    
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # 5. Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS DO FIT")
    print("=" * 60)
    
    if fit_result:
        result = fitter.Result()
        print(f"\nStatus do fit: {'Sucesso' if result.IsValid() else 'Falhou'}")
        print(f"Chi2/NDF: {result.Chi2():.4f} / {result.Ndf()}")
        print(f"Probabilidade: {result.Prob():.6f}")
        
        print("\nParâmetros ajustados:")
        for i in range(result.NPar()):
            print(f"  p[{i}] = {result.Parameter(i):.4f} ± {result.ParError(i):.4f}")
        
        # Atualizar a função TF1 com os parâmetros ajustados
        for i in range(result.NPar()):
            func.SetParameter(i, result.Parameter(i))
            func.SetParError(i, result.ParError(i))
        
    else:
        print("\nErro: O fit falhou!")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Exemplo de ajuste com 4 parâmetros livres usando ROOT::Fit::Fitter
com funções separadas e uso de fitter.LeastSquareFit(bin_data)
"""

import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo com 4 parâmetros livres
    f(x) = a1 * exp(-x/mg) + a2 * x^eps
    
    Args:
        x: array com coordenadas x
        par: array com parâmetros [mg, eps, a1, a2]
    
    Returns:
        Valor da função
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]
    
    return a1 * ROOT.TMath.Exp(-x[0] / mg) + a2 * (x[0] ** eps)

def fit_data(func_model, x_data, y_data, y_errors, initial_params, xmin, xmax):
    """
    Função exclusiva para realizar o ajuste usando LeastSquareFit
    
    Args:
        func_model: Função Python callable(x, par) a ser minimizada
        x_data: array com dados experimentais em x
        y_data: array com dados experimentais em y
        y_errors: array com erros em y
        initial_params: lista com valores iniciais dos parâmetros
        xmin: limite inferior do intervalo de ajuste
        xmax: limite superior do intervalo de ajuste
    
    Returns:
        dict: Dicionário com resultados do fit
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Definir parâmetros iniciais
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(0)
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados
    result = fitter.Result()
    
    output = {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'prob': result.Prob(),
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'valid': result.IsValid()
    }
    
    return output

def main():
    print("=" * 60)
    print("Ajuste com 4 parâmetros: mg, eps, a1, a2")
    print("=" * 60)
    
    # Gerar dados experimentais simulados
    np.random.seed(42)
    
    # Parâmetros verdadeiros
    true_mg = 2.0
    true_eps = 0.5
    true_a1 = 10.0
    true_a2 = 5.0
    
    # Gerar pontos x
    x_data = np.linspace(0.1, 10.0, 50)
    
    # Calcular y com a função verdadeira + ruído
    y_data = []
    y_errors = []
    
    for x in x_data:
        y_true = true_a1 * np.exp(-x / true_mg) + true_a2 * (x ** true_eps)
        noise = np.random.normal(0, 0.5 * np.sqrt(y_true))
        y_obs = y_true + noise
        y_err = 0.5 * np.sqrt(max(y_obs, 1.0))
        
        y_data.append(y_obs)
        y_errors.append(y_err)
    
    print(f"\nDados experimentais:")
    print(f"  Número de pontos: {len(x_data)}")
    print(f"\nParâmetros verdadeiros:")
    print(f"  mg  = {true_mg}")
    print(f"  eps = {true_eps}")
    print(f"  a1  = {true_a1}")
    print(f"  a2  = {true_a2}")
    
    # Parâmetros iniciais para o fit [mg, eps, a1, a2]
    initial_params = [1.5, 0.3, 8.0, 4.0]
    
    print(f"\nParâmetros iniciais:")
    print(f"  mg  = {initial_params[0]}")
    print(f"  eps = {initial_params[1]}")
    print(f"  a1  = {initial_params[2]}")
    print(f"  a2  = {initial_params[3]}")
    
    # Executar o ajuste
    print("\n" + "=" * 60)
    print("Executando ajuste...")
    print("=" * 60)
    
    fit_result = fit_data(
        func_model=model_function,
        x_data=x_data,
        y_data=y_data,
        y_errors=y_errors,
        initial_params=initial_params,
        xmin=x_data.min(),
        xmax=x_data.max()
    )
    
    # Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS DO AJUSTE")
    print("=" * 60)
    
    print(f"\nStatus: {'Sucesso' if fit_result['valid'] else 'Falhou'}")
    print(f"Chi²     = {fit_result['chi2']:.4f}")
    print(f"NDF      = {fit_result['ndf']}")
    print(f"Chi²/DOF = {fit_result['chi2_dof']:.4f}")
    print(f"Prob     = {fit_result['prob']:.6f}")
    
    param_names = ['mg', 'eps', 'a1', 'a2']
    print(f"\nParâmetros ajustados:")
    for i, name in enumerate(param_names):
        param = fit_result['parameters'][i]
        error = fit_result['errors'][i]
        print(f"  {name:3s} = {param:8.4f} ± {error:.4f}")
    
    print("\n" + "=" * 60)

if __name__ == "__main__":
    main()

In [ ]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo SIMPLES com 2 parâmetros: linha reta
    f(x) = a + b*x
    """
    a = par[0]
    b = par[1]
    return a + b * x[0]

def fit_data(func_model, x_data, y_data, y_errors, initial_params, xmin, xmax):
    """
    Função exclusiva para realizar o ajuste usando LeastSquareFit
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Definir parâmetros iniciais
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(0)
    fitter.Config().MinimizerOptions().SetStrategy(2)  
    fitter.Config().MinimizerOptions().SetTolerance(1e-4)
    fitter.Config().MinimizerOptions().SetPrecision(1e-4)

    # minimizer_options.SetTolerance(1e-4)           # Tolerância mais apertada
    # minimizer_options.SetPrecision(1e-8)           # Precisão numérica
    # minimizer_options.SetMaxFunctionCalls(10000)   # Máximo de chamadas da função
    # minimizer_options.SetMaxIterations(1000) 
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados
    result = fitter.Result()
    
    output = {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'prob': result.Prob(),
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'valid': result.IsValid()
    }
    
    return output

# =============================================
# EXEMPLO ULTRA SIMPLES - LINHA RETA
# =============================================

# Parâmetros verdadeiros (conhecidos)
true_params = {'a': 2.0, 'b': 1.5}

# Gerar dados simples - linha reta com ruído
np.random.seed(42)  # Para reproducibilidade
x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
y_true = true_params['a'] + true_params['b'] * x_data

# Adicionar ruído pequeno
noise = np.random.normal(0, 0.5, len(x_data))
y_data = y_true + noise

# Erros fixos (simples)
y_errors = np.full(len(x_data), 0.5)

# Parâmetros iniciais (chute razoável)
initial_params = [1.0, 2.0]  # [a, b]

# Executar o ajuste
result = fit_data(model_function, x_data, y_data, y_errors, initial_params, 1.0, 10.0)

# =============================================
# APRESENTAÇÃO DOS RESULTADOS
# =============================================

print("=" * 50)
print("TESTE ULTRA SIMPLES - AJUSTE LINHA RETA")
print("=" * 50)

print(f"\nPARÂMETROS VERDADEIROS:")
print(f"  a = {true_params['a']:.3f}")
print(f"  b = {true_params['b']:.3f}")

print(f"\nRESULTADOS DO AJUSTE:")
if result['valid']:
    print(f"  a = {result['parameters'][0]:.3f} ± {result['errors'][0]:.3f}")
    print(f"  b = {result['parameters'][1]:.3f} ± {result['errors'][1]:.3f}")
    
    print(f"\nQUALIDADE DO AJUSTE:")
    print(f"  Chi²      = {result['chi2']:.3f}")
    print(f"  Graus de liberdade = {result['ndf']}")
    print(f"  Chi²/DOF  = {result['chi2_dof']:.3f}")
    print(f"  Probabilidade = {result['prob']:.3f}")
    
    print(f"\nCOMPARAÇÃO COM VALORES REAIS:")
    diff_a = abs(result['parameters'][0] - true_params['a'])
    diff_b = abs(result['parameters'][1] - true_params['b'])
    
    print(f"  Diferença em a: {diff_a:.3f} ({diff_a/result['errors'][0]:.1f}σ)")
    print(f"  Diferença em b: {diff_b:.3f} ({diff_b/result['errors'][1]:.1f}σ)")
    
    # Verificação simples
    if diff_a < 2 * result['errors'][0] and diff_b < 2 * result['errors'][1]:
        print(f"\n✅ AJUSTE FUNCIONANDO CORRETAMENTE!")
        print(f"   Parâmetros dentro de 2σ dos valores verdadeiros")
    else:
        print(f"\n⚠️  Possível problema no ajuste")
        print(f"   Alguns parâmetros fora de 2σ")
        
else:
    print("❌ AJUSTE FALHOU!")

In [ ]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo SIMPLES com 2 parâmetros: linha reta
    f(x) = a + b*x
    """
    a = par[0]
    b = par[1]
    return a + b * x[0]

def fit_data(func_model, x_data, y_data, y_errors, initial_params, xmin, xmax):
    """
    Função exclusiva para realizar o ajuste usando LeastSquareFit
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Definir parâmetros iniciais
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(0)
    fitter.Config().MinimizerOptions().SetStrategy(2)  
    fitter.Config().MinimizerOptions().SetTolerance(1e-4)
    fitter.Config().MinimizerOptions().SetPrecision(1e-4)

    # minimizer_options.SetTolerance(1e-4)           # Tolerância mais apertada
    # minimizer_options.SetPrecision(1e-8)           # Precisão numérica
    # minimizer_options.SetMaxFunctionCalls(10000)   # Máximo de chamadas da função
    # minimizer_options.SetMaxIterations(1000) 
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados
    result = fitter.Result()
    
    output = {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'prob': result.Prob(),
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'valid': result.IsValid()
    }
    
    return output

def manual_chi2_dof(x_data, y_data, y_errors, params, model_func):
    """
    Calcula manualmente o Chi² e o Chi²/dof para comparação com o ROOT.

    Chi² = Σ [ (y_i - f(x_i; params))² / σ_i² ]
    dof = N - n_params
    """
    residuals = []
    chi2 = 0.0
    for i in range(len(x_data)):
        y_model = model_func(np.array([x_data[i]]), params)
        diff = y_data[i] - y_model
        chi2 += (diff ** 2) / (y_errors[i] ** 2)
        residuals.append(diff)
    
    n_points = len(x_data)
    n_params = len(params)
    dof = n_points - n_params
    
    chi2_dof = chi2 / dof if dof > 0 else float('nan')
    
    return chi2, dof, chi2_dof

# =============================================
# EXEMPLO ULTRA SIMPLES - LINHA RETA
# =============================================

# Parâmetros verdadeiros (conhecidos)
true_params = {'a': 2.0, 'b': 1.5}

# Gerar dados simples - linha reta com ruído
np.random.seed(42)  # Para reproducibilidade
x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
y_true = true_params['a'] + true_params['b'] * x_data

# Adicionar ruído pequeno
noise = np.random.normal(0, 0.5, len(x_data))
y_data = y_true + noise

# Erros fixos (simples)
y_errors = np.full(len(x_data), 0.5)

# Parâmetros iniciais (chute razoável)
initial_params = [1.0, 2.0]  # [a, b]

# Executar o ajuste
result = fit_data(model_function, x_data, y_data, y_errors, initial_params, 1.0, 10.0)

# =============================================
# APRESENTAÇÃO DOS RESULTADOS
# =============================================

print("=" * 50)
print("TESTE ULTRA SIMPLES - AJUSTE LINHA RETA")
print("=" * 50)

print(f"\nPARÂMETROS VERDADEIROS:")
print(f"  a = {true_params['a']:.3f}")
print(f"  b = {true_params['b']:.3f}")

print(f"\nRESULTADOS DO AJUSTE:")
if result['valid']:
    print(f"  a = {result['parameters'][0]:.3f} ± {result['errors'][0]:.3f}")
    print(f"  b = {result['parameters'][1]:.3f} ± {result['errors'][1]:.3f}")
    
    print(f"\nQUALIDADE DO AJUSTE:")
    print(f"  Chi²      = {result['chi2']:.3f}")
    print(f"  Graus de liberdade = {result['ndf']}")
    print(f"  Chi²/DOF  = {result['chi2_dof']:.3f}")
    print(f"  Probabilidade = {result['prob']:.3f}")
    
    print(f"\nCOMPARAÇÃO COM VALORES REAIS:")
    diff_a = abs(result['parameters'][0] - true_params['a'])
    diff_b = abs(result['parameters'][1] - true_params['b'])
    
    print(f"  Diferença em a: {diff_a:.3f} ({diff_a/result['errors'][0]:.1f}σ)")
    print(f"  Diferença em b: {diff_b:.3f} ({diff_b/result['errors'][1]:.1f}σ)")
    
    # Verificação simples
    if diff_a < 2 * result['errors'][0] and diff_b < 2 * result['errors'][1]:
        print(f"\n✅ AJUSTE FUNCIONANDO CORRETAMENTE!")
        print(f"   Parâmetros dentro de 2σ dos valores verdadeiros")
    else:
        print(f"\n⚠️  Possível problema no ajuste")
        print(f"   Alguns parâmetros fora de 2σ")
        
else:
    print("❌ AJUSTE FALHOU!")

# =============================================
# COMPARAÇÃO MANUAL DO CHI²/DOF
# =============================================
chi2_manual, dof_manual, chi2_dof_manual = manual_chi2_dof(
    x_data, y_data, y_errors, result['parameters'], model_function
)

print("\nCÁLCULO MANUAL DO CHI²:")
print(f"  Chi² manual    = {chi2_manual:.3f}")
print(f"  DOF manual     = {dof_manual}")
print(f"  Chi²/DOF manual= {chi2_dof_manual:.3f}")
print(f"\nDiferença relativa |Chi²/DOF|: {abs(result['chi2_dof'] - chi2_dof_manual)/result['chi2_dof']}")


In [1]:
import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo COMPLEXA com 4 parâmetros livres:
    f(x) = mg * exp(-eps * x[0]) + a1 * x[0] + a2 * x[0]**2
    
    par[0] = mg  (magnitude/amplitude)
    par[1] = eps (taxa de decaimento exponencial)
    par[2] = a1  (coeficiente linear)
    par[3] = a2  (coeficiente quadrático)
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]

    # Prevenir overflow no exp
    arg = eps * (x[0] - a1)
    if arg > 100:
        logistic = 0.0
    elif arg < -100:
        logistic = 1.0
    else:
        logistic = 1.0 / (1.0 + ROOT.TMath.Exp(arg))

    return mg * logistic + a2 * x[0]

def fit_data(func_model, x_data, y_data, y_errors, initial_params, param_limits, xmin, xmax):
    """
    Função para realizar o ajuste usando LeastSquareFit com ROOT
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Nomear os parâmetros
    param_names = ['mg', 'eps', 'a1', 'a2']
    for i, name in enumerate(param_names):
        func.SetParName(i, name)
    
    # Definir parâmetros iniciais e limites
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
        if param_limits[i] is not None:
            func.SetParLimits(i, param_limits[i][0], param_limits[i][1])
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    # Configurações do minimizador Minuit2
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(0)
    fitter.Config().MinimizerOptions().SetStrategy(2)
    fitter.Config().MinimizerOptions().SetPrecision(1e-8)  
    fitter.Config().MinimizerOptions().SetTolerance(1e-3)
    fitter.Config().MinimizerOptions().SetMaxFunctionCalls(50000)
    fitter.Config().MinimizerOptions().SetMaxIterations(50000)
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados
    result = fitter.Result()
    
    output = {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'param_names': param_names,
        'valid': result.IsValid(),
        'status': result.Status()
    }
    
    return output

# =============================================
# EXEMPLO COM FUNÇÃO COMPLEXA - 4 PARÂMETROS
# =============================================


true_params = {'mg': 8.0, 'eps': 1.2, 'a1': 5.0, 'a2': 0.3}
np.random.seed(42)
x_data = np.linspace(0.5, 10.0, 30)

# Dados simulados da função estável
y_true = np.array([model_function([x], list(true_params.values())) for x in x_data])

noise_level = 0.3
y_data = y_true + np.random.normal(0, noise_level, len(x_data))
y_errors = np.full(len(x_data), noise_level)

# Chutes iniciais razoáveis
initial_params = [5.0, 0.5, 4.0, 0.1]
param_limits = [
    (0.1, 50.0),     # mg
    (0.01, 10.0),    # eps
    (0.1, 10.0),     # a1
    (-2.0, 2.0)      # a2
]

# Executar o ajuste
result = fit_data(model_function, x_data, y_data, y_errors, initial_params, 
                  param_limits, x_data.min(), x_data.max())

# =============================================
# APRESENTAÇÃO SIMPLIFICADA DOS RESULTADOS
# =============================================

print("=" * 60)
print("RESULTADOS DO AJUSTE")
print("=" * 60)

if result['valid']:
    print(f"\nChi²/DOF = {result['chi2_dof']:.4f}\n")
    
    print("PARÂMETROS AJUSTADOS:")
    for i, name in enumerate(result['param_names']):
        param_val = result['parameters'][i]
        param_err = result['errors'][i]
        print(f"  {name:4s} = {param_val:8.4f} ± {param_err:.4f}")
else:
    print(f"❌ AJUSTE FALHOU! (Status: {result['status']})")
    print("\nParâmetros tentados:")
    for i, name in enumerate(result['param_names']):
        print(f"  {name:4s} = {result['parameters'][i]:8.4f}")
    
print("=" * 60)

RESULTADOS DO AJUSTE

Chi²/DOF = 0.6305

PARÂMETROS AJUSTADOS:
  mg   =   8.2104 ± 0.1410
  eps  =   1.1953 ± 0.0813
  a1   =   4.8639 ± 0.0665
  a2   =   0.2917 ± 0.0127


In [ ]:
import ROOT
import numpy as np
from typing import Callable, List, Dict, Any

# Example 1: Basic usage with TGraphErrors and Chisquare
def example_basic_chisquare():
    """Demonstrate basic usage of graph.Chisquare(func)"""
    
    # Create some data with errors
    x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=float)
    y_data = np.array([2.1, 4.2, 5.8, 8.1, 10.2], dtype=float)
    y_errors = np.array([0.3, 0.3, 0.4, 0.4, 0.5], dtype=float)
    
    # Create TGraphErrors
    graph = ROOT.TGraphErrors(len(x_data), x_data, y_data, 
                              np.zeros(len(x_data)), y_errors)
    
    # Define a fitting function (linear: f(x) = p0 + p1*x)
    func = ROOT.TF1("linear", "[0] + [1]*x", 0, 6)
    func.SetParameter(0, 0.0)  # intercept
    func.SetParameter(1, 2.0)  # slope
    
    # Calculate chi-square
    chi2 = graph.Chisquare(func)
    ndf = len(x_data) - 2  # 2 parameters
    
    print(f"Chi² = {chi2:.4f}")
    print(f"NDF = {ndf}")
    print(f"Chi²/NDF = {chi2/ndf:.4f}")
    
    return graph, func, chi2


# Example 2: Using with GenericMinimizer
def fit_with_graph_chisquare(x_data: np.ndarray, 
                             y_data: np.ndarray,
                             y_errors: np.ndarray,
                             func_formula: str,
                             n_params: int,
                             initial_params: List[float],
                             param_names: List[str] = None) -> Dict[str, Any]:
    """
    Fit data using ROOT's graph.Chisquare method with GenericMinimizer.
    
    Parameters:
    -----------
    x_data : np.ndarray
        X coordinates
    y_data : np.ndarray
        Y coordinates
    y_errors : np.ndarray
        Errors on Y values
    func_formula : str
        ROOT TF1 formula string (e.g., "[0] + [1]*x")
    n_params : int
        Number of parameters
    initial_params : List[float]
        Initial parameter values
    param_names : List[str], optional
        Parameter names
        
    Returns:
    --------
    Dict with fit results including chi2/ndf
    """
    
    # Create TGraphErrors
    graph = ROOT.TGraphErrors(len(x_data), x_data, y_data,
                              np.zeros(len(x_data)), y_errors)
    
    # Create function
    x_min, x_max = float(np.min(x_data)), float(np.max(x_data))
    func = ROOT.TF1("fit_func", func_formula, x_min, x_max)
    
    # Define chi2 function for minimizer
    def chi2_func(params):
        for i in range(n_params):
            func.SetParameter(i, params[i])
        return graph.Chisquare(func)
    
    # Import the GenericMinimizer (assuming it's available)
    # For this example, we'll do a simplified version
    minimizer = ROOT.Math.Factory.CreateMinimizer("Minuit2", "Migrad")
    minimizer.SetMaxFunctionCalls(1000000)
    minimizer.SetTolerance(0.001)
    minimizer.SetPrintLevel(1)
    
    # Set function
    minimizer_func = ROOT.Math.Functor(chi2_func, n_params)
    minimizer.SetFunction(minimizer_func)
    
    # Set variables
    if param_names is None:
        param_names = [f"p{i}" for i in range(n_params)]
    
    initial_steps = [abs(p * 0.1) if p != 0 else 0.1 for p in initial_params]
    for i in range(n_params):
        minimizer.SetVariable(i, param_names[i], initial_params[i], initial_steps[i])
    
    # Minimize
    minimizer.Minimize()
    
    # Get results
    best_params = [minimizer.X()[i] for i in range(n_params)]
    chi2_min = minimizer.MinValue()
    ndf = len(x_data) - n_params
    chi2_ndf = chi2_min / ndf if ndf > 0 else float('inf')
    prob = ROOT.TMath.Prob(chi2_min, ndf) if ndf > 0 else 0.0
    
    # Set best parameters to function
    for i in range(n_params):
        func.SetParameter(i, best_params[i])
    
    result = {
        'success': bool(minimizer.Status() == 0),
        'parameters': best_params,
        'parameter_errors': [minimizer.Errors()[i] for i in range(n_params)],
        'chi2': chi2_min,
        'ndf': ndf,
        'chi2_ndf': chi2_ndf,
        'prob': prob,
        'graph': graph,
        'function': func,
        'minimizer': minimizer
    }
    
    return result


In [7]:
import ROOT
import numpy as np
from typing import Callable, List, Tuple, Dict, Any, Optional

def GenericMinimizer(objective_func: Callable,
                    n_dim: int,
                    initial_params: List[float],
                    initial_steps: List[float],
                    param_names: Optional[List[str]] = None,
                    minimizer_name: str = "Minuit2",
                    algorithm_name: str = "",
                    max_iterations: int = 10000,
                    max_function_calls: int = 1000000,
                    tolerance: float = 0.001,
                    print_level: int = 1,
                    fixed_params: Optional[Dict[int, float]] = None,
                    lower_bounds: Optional[Dict[int, float]] = None,
                    upper_bounds: Optional[Dict[int, float]] = None,
                    random_seed: int = -1,
                    compute_hesse: bool = True,
                    compute_minos: bool = False,
                    minos_params: Optional[List[int]] = None) -> Dict[str, Any]:
    """
    Generic function for numerical minimization using ROOT's minimizer.
    
    Parameters:
    -----------
    objective_func : Callable
        The objective function to minimize. Should take a list of parameters and return a float.
    n_dim : int
        Number of parameters/dimensions.
    initial_params : List[float]
        Initial values for the parameters.
    initial_steps : List[float]  
        Initial step sizes for the parameters.
    param_names : Optional[List[str]]
        Names for the parameters. If None, will use "param_0", "param_1", etc.
    minimizer_name : str
        Name of the minimizer (e.g., "Minuit2", "Minuit", "GSLMultiMin")
    algorithm_name : str
        Specific algorithm to use (e.g., "Migrad", "Simplex", "BFGS")
    max_iterations : int
        Maximum number of iterations.
    max_function_calls : int
        Maximum number of function calls.
    tolerance : float
        Tolerance for convergence.
    print_level : int
        Print level (0 = quiet, 1 = normal, 2 = verbose).
    fixed_params : Optional[Dict[int, float]]
        Dictionary of parameter indices to fixed values.
    lower_bounds : Optional[Dict[int, float]]
        Dictionary of parameter indices to lower bounds.
    upper_bounds : Optional[Dict[int, float]]
        Dictionary of parameter indices to upper bounds.
    random_seed : int
        Random seed for initial parameter randomization (<0 for fixed initial params).
    compute_hesse : bool
        If True, compute Hessian matrix for accurate error estimation (default: True).
    compute_minos : bool
        If True, compute Minos asymmetric errors for better error estimates (default: False).
    minos_params : Optional[List[int]]
        List of parameter indices for which to compute Minos errors. 
        If None and compute_minos=True, compute for all free parameters.
        
    Returns:
    --------
    Dict[str, Any]
        Dictionary containing minimization results:
        - 'success': bool indicating if minimization was successful
        - 'minimum_value': float, the minimum function value found
        - 'parameters': list of optimized parameters
        - 'parameter_errors': list of parameter errors (parabolic/Hesse errors)
        - 'iterations': number of iterations performed
        - 'function_calls': number of function calls
        - 'status_code': minimizer status code
        - 'hesse_status': status of Hesse calculation (if computed)
        - 'covariance_matrix': covariance matrix as 2D list (if Hesse computed)
        - 'correlation_matrix': correlation matrix as 2D list (if Hesse computed)
        - 'minos_errors': dict with Minos errors {param_idx: (lower, upper)} (if computed)
        - 'minos_status': dict with Minos status for each parameter (if computed)
        - 'minimizer': the minimizer object for further analysis
    """
    
    # Create minimizer
    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizer_name, algorithm_name)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizer_name}\". " 
                          "Maybe the required library was not built?")
    
    # Set minimizer parameters
    minimizer.SetMaxFunctionCalls(max_function_calls)
    minimizer.SetMaxIterations(max_iterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetPrintLevel(print_level)
    minimizer.SetPrecision(1e-8)
    minimizer.SetStrategy(2)
    # Create function wrapper
    func = ROOT.Math.Functor(objective_func, n_dim)
    minimizer.SetFunction(func)
    
    # Handle random initial parameters
    variables = initial_params.copy()
    if random_seed >= 0:
        r = ROOT.TRandom2(random_seed)
        for i in range(n_dim):
            variables[i] = r.Uniform(-10, 10)  # Adjust range as needed
    
    # Set parameters
    if param_names is None:
        param_names = [f"param_{i}" for i in range(n_dim)]
    
    for i in range(n_dim):
        if fixed_params and i in fixed_params:
            # Fixed parameter
            minimizer.SetFixedVariable(i, param_names[i], fixed_params[i])
        elif lower_bounds and upper_bounds and i in lower_bounds and i in upper_bounds:
            # Bounded parameter
            minimizer.SetLimitedVariable(i, param_names[i], variables[i], 
                                       initial_steps[i], lower_bounds[i], upper_bounds[i])
        elif lower_bounds and i in lower_bounds:
            # Lower bound only
            minimizer.SetLowerLimitedVariable(i, param_names[i], variables[i],
                                            initial_steps[i], lower_bounds[i])
        elif upper_bounds and i in upper_bounds:
            # Upper bound only  
            minimizer.SetUpperLimitedVariable(i, param_names[i], variables[i],
                                            initial_steps[i], upper_bounds[i])
        else:
            # Free parameter
            minimizer.SetVariable(i, param_names[i], variables[i], initial_steps[i])
    
    # Perform minimization
    status = minimizer.Minimize()
    
    # Extract basic results
    result = {
        'success': bool(status),
        'minimum_value': minimizer.MinValue(),
        'parameters': [minimizer.X()[i] for i in range(n_dim)],
        'parameter_errors': [minimizer.Errors()[i] for i in range(n_dim)] if minimizer.Errors() else None,
        'iterations': minimizer.NIterations(),
        'function_calls': minimizer.NCalls(),
        'status_code': status,
        'minimizer': minimizer
    }
    
    # Compute Hesse errors (full error matrix calculation)
    if compute_hesse:
        if print_level > 0:
            print("\nComputing Hessian matrix for error analysis...")
        
        hesse_status = minimizer.Hesse()
        result['hesse_status'] = bool(hesse_status)
        
        if hesse_status:
            # Get covariance matrix
            cov_matrix = np.zeros((n_dim, n_dim))
            minimizer.GetCovMatrix(cov_matrix.flatten())
            result['covariance_matrix'] = cov_matrix.tolist()
            
            # Calculate correlation matrix from covariance
            corr_matrix = np.zeros((n_dim, n_dim))
            for i in range(n_dim):
                for j in range(n_dim):
                    if cov_matrix[i, i] > 0 and cov_matrix[j, j] > 0:
                        corr_matrix[i, j] = cov_matrix[i, j] / np.sqrt(cov_matrix[i, i] * cov_matrix[j, j])
                    else:
                        corr_matrix[i, j] = 0.0
            result['correlation_matrix'] = corr_matrix.tolist()
            
            # Update parameter errors with Hesse errors
            result['parameter_errors'] = [minimizer.Errors()[i] for i in range(n_dim)]
            
            if print_level > 0:
                print("Hesse calculation successful!")
        else:
            result['covariance_matrix'] = None
            result['correlation_matrix'] = None
            if print_level > 0:
                print("Warning: Hesse calculation failed!")
    
    # Compute Minos errors (asymmetric errors)
    if compute_minos:
        if print_level > 0:
            print("\nComputing Minos errors...")
        
        # Determine which parameters to compute Minos for
        if minos_params is None:
            # Compute for all free (non-fixed) parameters
            minos_params = [i for i in range(n_dim) 
                          if not (fixed_params and i in fixed_params)]
        
        minos_errors = {}
        minos_status = {}
        
        for param_idx in minos_params:
            err_low = np.zeros(1, dtype=np.float64)
            err_up = np.zeros(1, dtype=np.float64)
            
            # GetMinosError returns bool indicating success
            status = minimizer.GetMinosError(param_idx, err_low, err_up)
            
            minos_status[param_idx] = bool(status)
            if status:
                # Minos errors are typically negative for lower and positive for upper
                minos_errors[param_idx] = (float(err_low[0]), float(err_up[0]))
                
                if print_level > 0:
                    param_name = param_names[param_idx]
                    param_val = result['parameters'][param_idx]
                    print(f"  {param_name}: {param_val:.6f} {err_low[0]:+.6f} {err_up[0]:+.6f}")
            else:
                minos_errors[param_idx] = (None, None)
                if print_level > 0:
                    print(f"  Warning: Minos failed for parameter {param_names[param_idx]}")
        
        result['minos_errors'] = minos_errors
        result['minos_status'] = minos_status
        
        if print_level > 0 and all(minos_status.values()):
            print("Minos calculation successful for all requested parameters!")
    
    return result


def PrintMinimizationResults(result: Dict[str, Any], 
                            param_names: Optional[List[str]] = None,
                            n_data_points: Optional[int] = None):
    """
    Pretty print minimization results including Hesse and Minos errors.
    
    Parameters:
    -----------
    result : Dict[str, Any]
        Result dictionary from GenericMinimizer
    param_names : Optional[List[str]]
        Parameter names for display
    n_data_points : Optional[int]
        Number of data points (for chi2/ndf calculation)
    """
    n_params = len(result['parameters'])
    if param_names is None:
        param_names = [f"param_{i}" for i in range(n_params)]
    
    print("\n" + "="*70)
    print("MINIMIZATION RESULTS")
    print("="*70)
    
    print(f"\nStatus: {'SUCCESS' if result['success'] else 'FAILED'}")
    print(f"Minimum value: {result['minimum_value']:.6e}")
    print(f"Function calls: {result['function_calls']}")
    print(f"Iterations: {result['iterations']}")
    
    # Chi2/ndf if data points provided
    if n_data_points is not None:
        n_free = result['minimizer'].NFree()
        ndf = n_data_points - n_free
        if ndf > 0:
            chi2_ndf = result['minimum_value'] / ndf
            prob = ROOT.TMath.Prob(result['minimum_value'], ndf)
            print(f"\nChi²/ndf: {result['minimum_value']:.4f}/{ndf} = {chi2_ndf:.4f}")
            print(f"Probability: {prob:.4f}")
    
    print("\n" + "-"*70)
    print("PARAMETER VALUES AND ERRORS")
    print("-"*70)
    
    # Print parameters with all available error types
    for i, (name, val) in enumerate(zip(param_names, result['parameters'])):
        error_str = ""
        
        # Parabolic/Hesse error
        if result['parameter_errors']:
            error_str = f" ± {result['parameter_errors'][i]:.6f}"
        
        # Minos errors (asymmetric)
        if 'minos_errors' in result and i in result['minos_errors']:
            err_low, err_up = result['minos_errors'][i]
            if err_low is not None and err_up is not None:
                error_str += f"  (Minos: {err_low:+.6f} {err_up:+.6f})"
        
        print(f"{name:15s} = {val:12.6f}{error_str}")
    
    # Print correlation matrix if available
    if 'correlation_matrix' in result and result['correlation_matrix'] is not None:
        print("\n" + "-"*70)
        print("CORRELATION MATRIX")
        print("-"*70)
        corr = np.array(result['correlation_matrix'])
        
        # Print header
        print(f"{'':15s}", end="")
        for name in param_names:
            print(f"{name:>12s}", end="")
        print()
        
        # Print matrix
        for i, name in enumerate(param_names):
            print(f"{name:15s}", end="")
            for j in range(len(param_names)):
                print(f"{corr[i, j]:12.4f}", end="")
            print()
    
    print("="*70 + "\n")


# Example usage
if __name__ == "__main__":
    # Example: Rosenbrock function
    def rosenbrock(params):
        x, y = params[0], params[1]
        return (1 - x)**2 + 100 * (y - x**2)**2
    
    # print("Example 1: Basic minimization with Hesse")
    # result1 = GenericMinimizer(
    #     rosenbrock,
    #     n_dim=2,
    #     initial_params=[0.0, 0.0],
    #     initial_steps=[0.1, 0.1],
    #     param_names=["x", "y"],
    #     compute_hesse=True,
    #     compute_minos=False,
    #     print_level=1
    # )
    # PrintMinimizationResults(result1, param_names=["x", "y"])
    
    # print("\n" + "="*70)
    # print("Example 2: Minimization with both Hesse and Minos")
    # result2 = GenericMinimizer(
    #     rosenbrock,
    #     n_dim=2,
    #     initial_params=[0.0, 0.0],
    #     initial_steps=[0.1, 0.1],
    #     param_names=["x", "y"],
    #     compute_hesse=True,
    #     compute_minos=True,
    #     print_level=1
    # )
    # PrintMinimizationResults(result2, param_names=["x", "y"])
    
    # Example with chi2 fitting
    print("\n" + "="*70)
    print("Example 3: Chi-square fit with graph.Chisquare()")
    
    # Create some data
    x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=float)
    y_data = np.array([2.1, 4.2, 5.8, 8.1, 10.2], dtype=float)
    y_errors = np.array([0.3, 0.3, 0.4, 0.4, 0.5], dtype=float)
    
    graph = ROOT.TGraphErrors(len(x_data), x_data, y_data, 
                              np.zeros(len(x_data)), y_errors)
    func = ROOT.TF1("linear", "[0] + [1]*x", 0, 6)
    
    def chi2_func(params):
        func.SetParameter(0, params[0])
        func.SetParameter(1, params[1])
        return graph.Chisquare(func)
    
    result3 = GenericMinimizer(
        chi2_func,
        n_dim=2,
        initial_params=[0.0, 2.0],
        initial_steps=[0.1, 0.1],
        param_names=["intercept", "slope"],
        compute_hesse=True,
        compute_minos=True,
        print_level=1
    )
    PrintMinimizationResults(result3, param_names=["intercept", "slope"], 
                           n_data_points=len(x_data))


Example 3: Chi-square fit with graph.Chisquare()

Computing Hessian matrix for error analysis...
Hesse calculation successful!

Computing Minos errors...
  intercept: 0.108579 -0.343108 +0.343108
  slope: 1.992830 -0.120912 +0.120912
Minos calculation successful for all requested parameters!

MINIMIZATION RESULTS

Status: SUCCESS
Minimum value: 7.066749e-01
Function calls: 34
Iterations: 34

Chi²/ndf: 0.7067/3 = 0.2356
Probability: 0.8716

----------------------------------------------------------------------
PARAMETER VALUES AND ERRORS
----------------------------------------------------------------------
intercept       =     0.108579 ± 0.343108  (Minos: -0.343108 +0.343108)
slope           =     1.992830 ± 0.120912  (Minos: -0.120912 +0.120912)

----------------------------------------------------------------------
CORRELATION MATRIX
----------------------------------------------------------------------
                  intercept       slope
intercept            0.0000      0.0000

Warning in <Minuit2>: VariableMetricBuilder No improvement in line search
Warning in <Minuit2>: VariableMetricBuilder No improvement in line search
